# 1. Tutorial overview

This tutorial implements a five-agent radiology report-generation pipeline: Retrieval, Draft, Refiner, Vision, and Synthesis. It replaces private APIs and controlled patient data with open models, synthetic retrieval text, and public demonstration radiographs.

## Data substitution and its limitations

The original research workflow was developed around curated chest X-ray datasets and locally prepared retrieval resources. Such datasets may require registration, a data-use agreement, institutional approval, controlled storage, or restrictions on redistribution. Those requirements are appropriate for research, but they prevent a chapter companion from being opened and executed by every reader without additional credentials.

This tutorial therefore uses two openly licensed demonstration radiographs: one normal example and one pneumonia example. They are individual teaching images, not a statistically representative dataset, and they must not be used to estimate accuracy, fairness, robustness, or clinical generalization. The accompanying retrieval reports are synthetic and de-identified; they demonstrate retrieval mechanics rather than reproduce a clinical knowledge base. Replacing controlled research data with public examples improves reproducibility, but it also means that the outputs in this notebook are illustrative rather than comparable with the original research results.

## Model substitution and modularity

The original system depended on an API-hosted language model, a local LLaVA-Med checkpoint, and a task-specific retrieval checkpoint. This tutorial substitutes openly downloadable models so that no API key or private checkpoint is required. BiomedCLIP is used for retrieval, Qwen2.5-1.5B-Instruct for the Draft, Refiner, and Synthesis Agents, and SmolVLM-500M-Instruct for the Vision Agent.

These choices prioritize accessibility and executable code, not clinical performance. The text and vision models are general-purpose models, BiomedCLIP is used without task-specific fine-tuning, and none of these components is validated here as a medical device. All model choices are replaceable: a reader may substitute another image-text encoder in the Retrieval Agent, another text-generation model in generate_text(), or another vision-language model in generate_vision(), provided that the replacement preserves the same input-output interface. This modularity allows the tutorial pipeline to remain useful as open models, institutional resources, and licensing requirements change.

**Educational use only. This pipeline is not clinically validated and must not be used for patient care.**

# 2. Environment and dependency installation

Use a Google Colab T4 GPU. Dependencies are installed directly in the runtime; no requirements file, API key, or companion upload is needed.

In [ ]:
%pip install -q "transformers==4.49.0" "accelerate==1.4.0" "open-clip-torch==2.31.0" "timm==1.0.15" "pillow==11.1.0" "matplotlib==3.10.0" "requests==2.32.3"


# 3. Reproducibility settings


In [ ]:
from pathlib import Path
import json, random
import requests
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Self-contained workspace: no companion files are required.
ROOT = Path("/content/multi_agent_rrg_demo") if Path("/content").exists() else Path.cwd() / "multi_agent_rrg_demo"
ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"Workspace: {ROOT} | Device: {DEVICE} | Seed: {SEED}")
if DEVICE != "cuda":
    print("Warning: CPU execution is possible but slow; Colab T4 GPU is recommended.")


# 4. Downloading a public demonstration example

The controlled research dataset is not used. This cell downloads a CC0 normal PA radiograph and a public-domain CDC pneumonia radiograph from Wikimedia Commons at runtime.

In [ ]:
PUBLIC_IMAGES = {
    "normal_pa_chest_xray.jpg": "https://commons.wikimedia.org/wiki/Special:Redirect/file/Normal_posteroanterior_%28PA%29_chest_radiograph_%28X-ray%29.jpg",
    "pneumonia_right_upper_lobe.jpg": "https://commons.wikimedia.org/wiki/Special:Redirect/file/Pneumonia_x_ray.jpg",
}
image_dir = ROOT / "public_images"
image_dir.mkdir(exist_ok=True)
image_paths = []

# Wikimedia rejects Python's default urllib User-Agent with HTTP 403.
DOWNLOAD_HEADERS = {
    "User-Agent": "UNT-RRG-Chapter-Demo/1.0 (educational reproducibility notebook)"
}

for filename, url in PUBLIC_IMAGES.items():
    destination = image_dir / filename
    if not destination.exists() or destination.stat().st_size < 1024:
        print(f"Downloading {filename} ...")
        response = requests.get(
            url,
            headers=DOWNLOAD_HEADERS,
            timeout=90,
            allow_redirects=True,
        )
        response.raise_for_status()
        content_type = response.headers.get("Content-Type", "").lower()
        if "image" not in content_type:
            raise RuntimeError(
                f"Expected an image for {filename}, received {content_type or 'unknown content type'}."
            )
        destination.write_bytes(response.content)
    image_paths.append(destination)

# Opening with PIL also verifies that the downloaded bytes are valid images.
images = [Image.open(path).convert("RGB") for path in image_paths]
fig, axes = plt.subplots(1, len(images), figsize=(10, 6))
for ax, image, path in zip(axes, images, image_paths):
    ax.imshow(image, cmap="gray")
    ax.set_title(path.stem.replace("_", " "))
    ax.axis("off")
plt.tight_layout()


# 5. Loading models and retrieval resources

BiomedCLIP performs retrieval, Qwen2.5-1.5B-Instruct supports the text agents, and SmolVLM-500M-Instruct supports the Vision Agent. The synthetic retrieval collection and all four prompts are embedded below so this Notebook is a single-file artifact.

## Why these models are used

The selected checkpoints can be downloaded without an API key and are small enough for a typical Colab GPU. They make the tutorial reproducible, but they are demonstration substitutes for the original task-specific components:

- **BiomedCLIP** supplies biomedical image-text embeddings, but this notebook does not fine-tune it on the demonstration task.
- **Qwen2.5-1.5B-Instruct** is a general instruction-following language model, not a specialist radiology reporting system.
- **SmolVLM-500M-Instruct** is a compact general-purpose vision-language model, not a clinically validated chest X-ray interpreter.

## Replaceable component interfaces

The pipeline intentionally isolates model-specific code behind three interfaces:

- retrieval_agent(image, top_k) returns ranked reference reports.
- generate_text(prompt) returns text for the Draft, Refiner, and Synthesis Agents.
- generate_vision(image, prompt) returns image-grounded findings for the Vision Agent.

A different open model, locally hosted institutional model, or approved API can replace any component without redesigning the remaining agents, as long as it returns the same type of output. Model substitutions should be documented with checkpoint revision, license, prompt changes, hardware requirements, and any new validation results.

In [ ]:
import open_clip
from transformers import AutoModelForCausalLM, AutoModelForVision2Seq, AutoProcessor, AutoTokenizer

CLIP_ID = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
TEXT_ID = "Qwen/Qwen2.5-1.5B-Instruct"
VISION_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# Synthetic, de-identified teaching reports. No patient dataset is used.
report_db = [
    {"id":"demo-normal-01","report":"Cardiomediastinal silhouette is within normal limits. Lungs are clear. No focal airspace opacity, pleural effusion, or pneumothorax."},
    {"id":"demo-normal-02","report":"Heart size is normal. No focal pulmonary opacity or pleural abnormality is identified."},
    {"id":"demo-normal-03","report":"The lungs are clear bilaterally. No acute cardiopulmonary abnormality is evident."},
    {"id":"demo-rul-01","report":"Focal airspace opacity is present in the right upper lung, suspicious for pneumonia. No pleural effusion."},
    {"id":"demo-rul-02","report":"Right upper lobe consolidation is present. Cardiomediastinal silhouette is not enlarged."},
    {"id":"demo-rul-03","report":"Patchy right upper lung opacity may represent an infectious airspace process. No pneumothorax."},
    {"id":"demo-bibasilar-01","report":"Mild bibasilar linear opacities favor subsegmental atelectatic change. No focal lobar consolidation."},
    {"id":"demo-edema-01","report":"Cardiac silhouette is enlarged with bilateral interstitial pulmonary opacities and small pleural effusions."},
]

# All prompts are centralized here so the single Notebook remains inspectable.
prompts = {
    "image_agent": """You are the Vision Agent in an educational chest X-ray report-generation pipeline.

Inspect only the supplied radiograph. Describe visible findings in at most three concise radiology-style sentences. Address lungs, pleura, and cardiomediastinal silhouette when visible. Do not infer history, compare with unavailable studies, or claim diagnostic certainty. If image quality limits assessment, say so.

Return plain text only, without headings or bullets.""",
    "general_agent": """You are the Draft Agent in an educational chest X-ray report-generation pipeline.

Create a preliminary report using only the retrieved reference reports below. Identify findings that are repeated or strongly supported. Do not assume that a retrieved case is identical to the query image. Use concise radiology-style sentences, no headings, and no bullets. Do not add facts absent from the retrieved evidence.

Retrieved reference reports:
{retrieved_reports}

Return only the preliminary report.""",
    "critical_agent": """You are the Refiner Agent in an educational chest X-ray report-generation pipeline.

Audit the preliminary report against the retrieved evidence. Retain only clinically important statements that are directly supported. Remove unsupported specificity and state uncertainty when the retrieved reports conflict. Use one concise paragraph without headings or bullets.

Preliminary report:
{preliminary_report}

Retrieved reference reports:
{retrieved_reports}

Return only the audited critical findings.""",
    "summarizing_agent": """You are the Synthesis Agent in an educational chest X-ray report-generation pipeline.

Synthesize a final report from the visual evidence and retrieval-grounded text. Prefer findings supported by both sources. When they disagree, prioritize the Vision Agent for visible anatomy but use cautious language; do not silently copy a retrieved diagnosis. Do not introduce new findings.

Vision Agent evidence:
{image_caption}

Draft report:
{preliminary_report}

Refined retrieval findings:
{critical_text}

Write exactly:
FINDINGS:
<concise findings>

IMPRESSION:
<one concise impression>""",
}

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(CLIP_ID)
clip_tokenizer = open_clip.get_tokenizer(CLIP_ID)
clip_model = clip_model.to(DEVICE).eval()

text_tokenizer = AutoTokenizer.from_pretrained(TEXT_ID)
text_model = AutoModelForCausalLM.from_pretrained(
    TEXT_ID, torch_dtype=DTYPE, low_cpu_mem_usage=True
).to(DEVICE).eval()

vision_processor = AutoProcessor.from_pretrained(VISION_ID)
vision_model = AutoModelForVision2Seq.from_pretrained(
    VISION_ID, torch_dtype=DTYPE, low_cpu_mem_usage=True, _attn_implementation="eager"
).to(DEVICE).eval()

def generate_text(prompt, max_new_tokens=180):
    messages = [
        {"role":"system", "content":"Educational radiology pipeline; not clinical care."},
        {"role":"user", "content":prompt},
    ]
    rendered = text_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = text_tokenizer(rendered, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        output = text_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output[0, inputs["input_ids"].shape[1]:]
    return text_tokenizer.decode(generated, skip_special_tokens=True).strip()

def generate_vision(image, prompt, max_new_tokens=160):
    messages = [{"role":"user", "content":[{"type":"image"}, {"type":"text", "text":prompt}]}]
    rendered = vision_processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = vision_processor(text=rendered, images=[image], return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        output = vision_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output[:, inputs["input_ids"].shape[1]:]
    return vision_processor.batch_decode(generated, skip_special_tokens=True)[0].strip()

print(f"Loaded {len(report_db)} synthetic reports and {len(prompts)} embedded prompts.")


# 6. Retrieval Agent

The Retrieval Agent embeds a query radiograph and candidate reports, then returns the highest cosine-similarity matches.

In [ ]:
def retrieval_agent(image, top_k=3):
    texts = [item["report"] for item in report_db]
    with torch.inference_mode():
        text_features = clip_model.encode_text(clip_tokenizer(texts).to(DEVICE), normalize=True)
        image_tensor = clip_preprocess(image).unsqueeze(0).to(DEVICE)
        image_features = clip_model.encode_image(image_tensor, normalize=True)
        scores, indices = (image_features @ text_features.T)[0].topk(top_k)
    return [{**report_db[i], "similarity":round(float(s), 4)}
            for s, i in zip(scores.cpu(), indices.cpu().tolist())]

retrieval_example = retrieval_agent(images[0])
retrieval_example

# 7. Draft Agent

The Draft Agent converts retrieved evidence into a preliminary report without seeing the query image.

In [ ]:
def draft_agent(retrieved_reports):
    evidence = "\n".join("- " + item["report"] for item in retrieved_reports)
    return generate_text(prompts["general_agent"].format(retrieved_reports=evidence))

draft_example = draft_agent(retrieval_example)
print(draft_example)

# 8. Refiner Agent

The Refiner Agent audits the draft against retrieved evidence and removes unsupported specificity.

In [ ]:
def refiner_agent(draft, retrieved_reports):
    evidence = "\n".join("- " + item["report"] for item in retrieved_reports)
    prompt = prompts["critical_agent"].format(
        preliminary_report=draft, retrieved_reports=evidence
    )
    return generate_text(prompt)

refined_example = refiner_agent(draft_example, retrieval_example)
print(refined_example)

# 9. Vision Agent

The Vision Agent independently describes visible image evidence and does not receive retrieved reports.

In [ ]:
def vision_agent(image):
    return generate_vision(image, prompts["image_agent"])

vision_example = vision_agent(images[0])
print(vision_example)

# 10. Synthesis Agent

The Synthesis Agent combines retrieval-grounded text and direct image evidence. Its prompt prevents silently copying a retrieved diagnosis when visual evidence disagrees.

In [ ]:
def synthesis_agent(draft, refined, vision):
    prompt = prompts["summarizing_agent"].format(
        image_caption=vision, preliminary_report=draft, critical_text=refined
    )
    return generate_text(prompt, max_new_tokens=220)

synthesis_example = synthesis_agent(draft_example, refined_example, vision_example)
print(synthesis_example)

# 11. Complete end-to-end execution

The function below is the single public entry point. It invokes all five agents and returns every intermediate output.

In [ ]:
def run_pipeline(image_path, top_k=3):
    image_path = Path(image_path)
    image = Image.open(image_path).convert("RGB")
    retrieved = retrieval_agent(image, top_k=top_k)
    draft = draft_agent(retrieved)
    refined = refiner_agent(draft, retrieved)
    vision = vision_agent(image)
    final = synthesis_agent(draft, refined, vision)
    return {
        "image":image_path.name,
        "retrieved_reports":retrieved,
        "draft_report":draft,
        "refined_report":refined,
        "vision_findings":vision,
        "final_report":final,
    }

In [ ]:
# Complete call: all five agents run without edits or manual intermediate steps.
pipeline_results = [run_pipeline(path, top_k=3) for path in image_paths]
output_path = ROOT / "outputs/pipeline_results.json"
output_path.parent.mkdir(exist_ok=True)
output_path.write_text(json.dumps(pipeline_results, indent=2, ensure_ascii=False))
print(f"Completed {len(pipeline_results)} cases. Saved: {output_path}")

# 12. Intermediate-output inspection


In [ ]:
for result in pipeline_results:
    print("\n" + "=" * 88 + "\n" + result["image"])
    for key in ["retrieved_reports","draft_report","refined_report","vision_findings","final_report"]:
        value = result[key]
        if key == "retrieved_reports":
            value = "\n".join(f"{x['similarity']:+.4f} | {x['report']}" for x in value)
        print(f"\n{key.upper()}\n{value}")

# 13. Automatic evaluation

These checks evaluate software execution, output structure, retrieval count, and expected demonstration concepts. They are not clinical validation.

In [ ]:
expected_terms = {
    "normal_pa_chest_xray.jpg":["clear","normal","no focal"],
    "pneumonia_right_upper_lobe.jpg":["opacity","consolidation","pneumonia"],
}
evaluation = []
for result in pipeline_results:
    final_lower = result["final_report"].lower()
    evaluation.append({
        "image":result["image"],
        "five_agents_completed":all(result[k] for k in [
            "retrieved_reports","draft_report","refined_report","vision_findings","final_report"
        ]),
        "retrieval_count_is_3":len(result["retrieved_reports"]) == 3,
        "has_findings_and_impression":"findings" in final_lower and "impression" in final_lower,
        "expected_demo_term_present":any(t in final_lower for t in expected_terms[result["image"]]),
    })
evaluation

# 14. Conflict-handling example

This controlled example intentionally conflicts: retrieval suggests right-upper-lobe pneumonia while visual evidence says no focal opacity. It demonstrates the synthesis policy without making a real diagnosis.

In [ ]:
conflict_draft = "Right upper lobe airspace opacity is suspicious for pneumonia."
conflict_refined = (
    "Retrieved reports support a right upper lung opacity, "
    "but retrieval alone cannot establish the query-image finding."
)
conflict_vision = "No focal airspace opacity is visible. Lungs appear clear."
conflict_output = synthesis_agent(conflict_draft, conflict_refined, conflict_vision)
print("RETRIEVAL-GROUNDED DRAFT:\n", conflict_draft)
print("\nCONFLICTING VISION EVIDENCE:\n", conflict_vision)
print("\nSYNTHESIS OUTPUT:\n", conflict_output)

# 15. Summary and suggested exercises

You executed a transparent five-agent pipeline with open resources and preserved every intermediate artifact.

Suggested exercises:

1. Change top_k and inspect retrieval sensitivity.
2. Compare BiomedCLIP with a general-domain CLIP checkpoint.
3. Modify one prompt at a time and record output changes.
4. Add a cardiomegaly conflict example.
5. Replace the synthetic collection with an openly licensed teaching collection.
6. Add expert-reviewed evaluation only after appropriate governance and ethics review.

For permanent citation, archive a versioned release in the UNT Data Repository or Zenodo and cite its DOI.